In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [31]:
block_size = 8
batch_size = 4
max_iters = 10000
#eval_interval = 250
learning_rate = 3e-4
eval_iters = 250

In [32]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8')as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [33]:
strng_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_strng = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [strng_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_strng[i] for i in l])

# encoded_hello = torch.tensor(encode('hello'),dtype=torch.long)
# decoded_hello = decode(encoded_hello.tolist())

data = torch.tensor(encode(text), dtype=torch.long)

In [34]:
n = int(0.8*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x= torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[62, 73, 58, 57,  9,  1, 54, 67],
        [57,  1, 73, 61, 54, 67, 64, 59],
        [72,  0, 76, 58, 71, 58,  1, 56],
        [68, 71, 68, 73, 61, 78,  9,  1]])
targets:
tensor([[73, 58, 57,  9,  1, 54, 67, 57],
        [ 1, 73, 61, 54, 67, 64, 59, 74],
        [ 0, 76, 58, 71, 58,  1, 56, 54],
        [71, 68, 73, 61, 78,  9,  1, 67]])


In [35]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is:',context,'target is:',target)

When input is: tensor([28]) target is: tensor(39)
When input is: tensor([28, 39]) target is: tensor(42)
When input is: tensor([28, 39, 42]) target is: tensor(39)
When input is: tensor([28, 39, 42, 39]) target is: tensor(44)
When input is: tensor([28, 39, 42, 39, 44]) target is: tensor(32)
When input is: tensor([28, 39, 42, 39, 44, 32]) target is: tensor(49)
When input is: tensor([28, 39, 42, 39, 44, 32, 49]) target is: tensor(1)
When input is: tensor([28, 39, 42, 39, 44, 32, 49,  1]) target is: tensor(25)


In [27]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [28]:
class BigramLanguagemodel(nn.Module):
    def __init__(self, vocab_size,hidden_size=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    #hidden layers
    

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index) #is 3 dim ->(B,T,C)

        # ---- Compute loss only if targets are provided (training mode) ----
        if targets is None:
            loss = None
        else:
            # Flatten both tensors to feed into cross_entropy
            B,T,C = logits.shape  #T "time" is the sequence size or block_size, C "channel" is vocab_size
            logits = logits.view(B*T, C) # B*T act as total number of samples, C class scores
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #cross_Entropy expects input:(N,C), targets:(N,)
            
        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get predictions
            logits, loss = self.forward(index)
            #for generation (not training) targets is None so skips logits flattening -> logtis become (B,T,C)
            logits = logits[:,-1,:] #becomes (B, C), -1 gets only the last token from the time step
            #apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C), dim=-1 acts across C class channels
            #sample for distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1), concatenates across time sequence, if dim=0 it would stack batches
        return index


model = BigramLanguagemodel(vocab_size)
m = model.to(device) 

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


.)]t,
f*[E.Hw9p2-I;S8T[-"yg:,dw4e(BH:u6cneN8kKIPDB"OQ f9:fXO2qk_rmrYXS,wu7"[rak]nG'gsR"2q8L3T("1B(3cz47F.4"WFgJdhovgDMNd?Wcng'1Nn3,DTYwgBby&G7S00J1C1BT(WA92qMtRazthD*I&cx,]C(9_"[H3EV1Bq4cz-S"y_FBTr_2!lIdY?Y7QJ7Q;nOB;9r3T
RMp29[,A RU4:J5lH(ybpLC"uAl&bQRTT(x ktvuJd
d"9KF':R_Cg7(.]WlI:',X"o:]a6lTYHFBqSLlNu4HBE[UixXTL"mCVj[".a,m'K
xgHDQ_F-M?admK]YHjbDkHQ
sCR[FVlIjna[uS8rRh4p-IK y_NOWeQ_d-qvj"Yv0,]slIdkaxC5DC!1NEQ(x]27cFsUXtaO C*;6,bQJ1HEuw7(lIDY[p
dXlEc5hj[6270?OYmW*_3T(dhqMzEc-?a3j]q"X1E!4CV1bQv&oI


In [29]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters ==0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
        
    #sample a batch of data
    xb,yb = get_batch("train")

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)#make grad star overnot accumulated
    loss.backward()
    optimizer.step()
    
print(loss.item())

step: 0, train loss: 4.9630, val loss: 4.9499
step: 250, train loss: 4.8482, val loss: 4.8385
step: 500, train loss: 4.7448, val loss: 4.7365
step: 750, train loss: 4.6355, val loss: 4.6294
step: 1000, train loss: 4.5327, val loss: 4.5280
step: 1250, train loss: 4.4375, val loss: 4.4319
step: 1500, train loss: 4.3387, val loss: 4.3384
step: 1750, train loss: 4.2472, val loss: 4.2452
step: 2000, train loss: 4.1561, val loss: 4.1579
step: 2250, train loss: 4.0722, val loss: 4.0727
step: 2500, train loss: 3.9881, val loss: 3.9878
step: 2750, train loss: 3.9043, val loss: 3.9151
step: 3000, train loss: 3.8276, val loss: 3.8357
step: 3250, train loss: 3.7525, val loss: 3.7630
step: 3500, train loss: 3.6798, val loss: 3.6907
step: 3750, train loss: 3.6110, val loss: 3.6251
step: 4000, train loss: 3.5465, val loss: 3.5604
step: 4250, train loss: 3.4860, val loss: 3.4974
step: 4500, train loss: 3.4264, val loss: 3.4405
step: 4750, train loss: 3.3721, val loss: 3.3848
step: 5000, train loss: 3.

In [30]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


i
B; meancaLidswhunor eglofeve aum lmacer antltitatJiei[(Le3-IN
Th Kt]tr Kter
dzade que6(07couplI coimUBangg d?weis!HMy. tJctitheavSo8Ly;cUng alis. thirndugs br kthin.d cifies I?"A, nem aqus t mermedXBincce I il;5, Ozay anthesor4( t aie
"YwGatou8Vathulancksar zary wit8T  " s Bur."wher3IF--s. hhiPAROTlepQquctstjuth?; od
Isthsey jut and that lT(y fo ewie omit, mequr r Wit wen9.

,0

ugrd,e, :
dt wid wath anggBtathRowindenhe
98Ld
bely, *J6swTh ered VY0dig,Dornengvx4edotha ad?2HQ; p-Abo'srou "bughy 
